In [65]:
import pandas as pd
import torch

# split into train and test
from sklearn.model_selection import train_test_split

device = (
    torch.accelerator.current_accelerator().type
    if torch.accelerator.is_available()
    else "cpu"
)
print(f"Using {device} device")

Using mps device


In [66]:
# load csv
# ./data/digit-recognizer/train.csv

df = pd.read_csv("../data/digit-recognizer/train.csv")

In [67]:
df.head()

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,1,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,4,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [68]:
# assert labels are values from 0 to 9
assert df["label"].isin(range(10)).all()
# assert pixels are values from 0 to 255
for col in df.columns[1:]:
    assert df[col].isin(range(256)).all()

# change values to boolean true if pixel value is greater than 128, false otherwise
for col in df.columns[1:]:
    df[col] = df[col] > 64
    assert df[col].isin([True, False]).all()

# chang columns to boolean
for col in df.columns[1:]:
    df[col] = df[col].astype(bool)
df

,label,pixel0,pixel1,pixel2,pixel3,pixel4,pixel5,pixel6,pixel7,pixel8,...,pixel774,pixel775,pixel776,pixel777,pixel778,pixel779,pixel780,pixel781,pixel782,pixel783
0,1,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
1,0,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
2,1,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
3,4,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
4,0,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
41995,0,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
41996,1,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
41997,7,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False
41998,6,False,False,False,False,False,False,False,False,False,...,False,False,False,False,False,False,False,False,False,False


In [ ]:
# create neural network model using torch
class DigitClassifier(torch.nn.Module):
    def __init__(self, input_size: int, hidden_layer_size: int, output_size: int)-> None:
        super(DigitClassifier, self).__init__()
        self.input_to_hidden_layer = torch.nn.Linear(input_size, hidden_layer_size)
        self.hidden_to_output_layer = torch.nn.Linear(hidden_layer_size, output_size)

    def forward(self, input_tensor: torch.Tensor) -> torch.Tensor:
        z1 = self.input_to_hidden_layer(input_tensor)
        a1 = torch.relu(z1)
        z2 = self.hidden_to_output_layer(a1)
        a2 = torch.softmax(z2, dim=1)
        return a2


X_train, X_test, y_train, y_test = train_test_split(
    df.drop("label", axis=1), df["label"], test_size=0.2, random_state=42
)

# convert to torch tensors
X_train_tensor = torch.tensor(X_train.values, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)
X_test_tensor = torch.tensor(X_test.values, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.long)
# create data loaders
train_dataset = torch.utils.data.TensorDataset(X_train_tensor, y_train_tensor)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=256, shuffle=True)

# Create the model
# input size is 784 (28x28 pixels), hidden size is 128, output size is 10 (digits from 0 to 9)
model = DigitClassifier(input_size=784, hidden_layer_size=128, output_size=10)
# train the model
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

model.train()
# train the model for 1000 epochs
for epoch in range(100):
    # forward pass
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        # make predictions
        y_hat = model(X_batch)
        # calculate loss
        loss = criterion(y_hat, y_batch)
        # calculate gradients
        loss.backward()
        # update weights
        optimizer.step()

    if (epoch + 1) % 10 == 0:
        accuracy = (y_hat.argmax(dim=1) == y_batch).sum().item() / len(y_batch)
        precision = (y_hat.argmax(dim=1)[y_batch == 1] == 1).sum().item() / (y_hat.argmax(dim=1) == 1).sum().item()
        recall = (y_hat.argmax(dim=1)[y_batch == 1] == 1).sum().item() / (y_batch == 1).sum().item()
        print(f"Epoch [{epoch+1}/100], Loss: {loss.item():.4f}, Accuracy: {accuracy:.6f}, Precision: {precision:.6f}, Recall: {recall:.6f}")


# evaluate the model
model.eval()
with torch.no_grad():
    y_hat = model(X_test_tensor)
    _, predicted = torch.max(y_hat.data, 1)
    accuracy = (predicted == y_test_tensor).sum().item() / len(y_test_tensor)
    precision = (predicted[y_test_tensor == 1] == 1).sum().item() / (predicted == 1).sum().item()
    recall = (predicted[y_test_tensor == 1] == 1).sum().item() / (y_test_tensor == 1).sum().item()
    print(f"Evaluation - Accuracy: {accuracy:.6f}, Precision: {precision:.6f}, Recall: {recall:.6f}")

Epoch [10/100], Loss: 1.5027, Accuracy: 0.968750, Precision: 1.000000, Recall: 1.000000
Epoch [20/100], Loss: 1.4718, Accuracy: 0.984375, Precision: 1.000000, Recall: 1.000000
Epoch [30/100], Loss: 1.4939, Accuracy: 0.968750, Precision: 1.000000, Recall: 1.000000
Epoch [40/100], Loss: 1.4929, Accuracy: 0.968750, Precision: 1.000000, Recall: 1.000000
Epoch [50/100], Loss: 1.4771, Accuracy: 0.984375, Precision: 1.000000, Recall: 1.000000
Epoch [60/100], Loss: 1.4613, Accuracy: 1.000000, Precision: 1.000000, Recall: 1.000000
Epoch [70/100], Loss: 1.4766, Accuracy: 0.984375, Precision: 1.000000, Recall: 1.000000
Epoch [80/100], Loss: 1.4768, Accuracy: 0.984375, Precision: 1.000000, Recall: 1.000000
Epoch [90/100], Loss: 1.5073, Accuracy: 0.953125, Precision: 1.000000, Recall: 1.000000
Epoch [100/100], Loss: 1.4612, Accuracy: 1.000000, Precision: 1.000000, Recall: 1.000000
Accuracy: 0.967500, Precision: 0.985699, Recall: 0.985699


In [73]:
# build sample submission file from the example submission file ./data/digit-recognizer/sample_submission.csv
sample_submission = pd.read_csv("../data/digit-recognizer/sample_submission.csv")
sample_submission.head()


,ImageId,Label
0,1,0
1,2,0
2,3,0
3,4,0
4,5,0


In [74]:
# ../data/digit-recognizer/test.csv
test_df = pd.read_csv("../data/digit-recognizer/test.csv")
test_df.head()

# make predictions for the test set
test_tensor = torch.tensor(test_df.values, dtype=torch.float32)
with torch.no_grad():
    y_hat = model(test_tensor)
    _, predicted = torch.max(y_hat.data, 1)
# create submission file
submission = pd.DataFrame(
    {"ImageId": range(1, len(predicted) + 1), "Label": predicted.numpy()}
)
submission.to_csv("../data/digit-recognizer/submission.csv", index=False)


In [ ]:
# !kaggle competitions submit -c digit-recognizer -f ../data/digit-recognizer/submission.csv -m "Message"

100%|█████████████████████████████████████████| 208k/208k [00:00<00:00, 292kB/s]
Successfully submitted to Digit Recognizer